In [1]:
# IN06: Failure Resilience

In [2]:
# Problem Statement:
#     How do we make a Walmart AI agent reliable when the external systems it depends on are slow, wrong, or temporarily unavailable?

# It handles this using three production problems:

# 1. Timeout + Retry
# 2. Hallucination Detection
# 3. Circuit Breaker

# So the overall idea is:
#     Don’t assume tools and APIs will always work. Build the agent so that it can timeout safely, retry intelligently, validate LLM output before taking action, and temporarily stop calling unhealthy services.

In [5]:
import os, json, time, random
from pathlib import Path
from dotenv import load_dotenv

from langchain_core.messages import HumanMessage, AIMessage, SystemMessage, ToolMessage
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode
from typing import TypedDict, Annotated

load_dotenv(override=True)
OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')
llm = ChatOpenAI(model='gpt-4-turbo', api_key=OPENAI_API_KEY, temperature=0)
print('LLM ready:', llm.model_name)

LLM ready: gpt-4-turbo


## Failure Scenarios and Resilience Patterns

Agentic systems fail in ways that traditional software does not.
Three failure modes every production agent must handle:

| Failure Mode | What happens | Detection | Mitigation |
|---|---|---|---|
| **Timeout** | External tool or LLM call exceeds latency budget | `time.time()` comparison or `signal.alarm` | Retry with back-off; fallback to cached result |
| **Hallucination** | LLM returns plausible but incorrect tool arguments or facts | Check tool output against known schema; LLM-as-judge | Re-prompt; return 'I don't know'; escalate to human |
| **Lost context** | Message history exceeds context window; older messages truncated | Token count check before invoke | Summarise history; use external memory store |

In [6]:
# Timeout:
# Try once → wait 1 second → try again → wait 2 seconds → try again.
# This is called back-off.
# If it still fails, we may return the last cached inventory information.

In [7]:
# Timeout → the tool is taking too long.
# Hallucination → the model makes something up.
# Lost context → the model forgets older information because the conversation became too large.

# Timeout → retry safely. Hallucination → validate before acting. Lost context → summarize and store important memory.

### Failure Mode 1: Timeout with Exponential Back-off Retry

In [ ]:
# If an external tool is too slow, don’t let the agent wait forever. Stop waiting after a fixed timeout, retry a few times with increasing delay, and if it still fails, use a fallback.

import threading

class ToolTimeoutError(Exception):
    pass

def call_with_timeout(fn, args: dict, timeout_sec: float = 2.0):
    result = [None]
    error  = [None]

    def target():
        try:
            result[0] = fn(**args) # slow_inventory_api(sku='GV-MILK-1G')
        except Exception as e:
            error[0] = e

    t = threading.Thread(target=target, daemon=True)
    t.start()
    t.join(timeout=timeout_sec) # Wait only for the timeout period
    if t.is_alive():
        raise ToolTimeoutError(f'Tool call timed out after {timeout_sec}s')
    if error[0]:
        raise error[0]
    return result[0]

def retry_with_backoff(fn, args: dict, max_retries: int = 3, base_delay: float = 0.5):
    last_error = None
    for attempt in range(max_retries):
        try:
            return call_with_timeout(fn, args, timeout_sec=2.0)
        except ToolTimeoutError as e:
            last_error = e
            delay = base_delay * (2 ** attempt)
            # Suppose base_delay = 0.5
            # Attempt1:
            # 0.5 × 2⁰ = 0.5 seconds
            # Attempt2:
            # 0.5 × 2¹ = 1.0 seconds
            # Attempt3:
            # 0.5 × 2² = 2.0 seconds
            # So instead of hammering the failing service continuously:
            # we progressively give it more time to recover.
            # That is exponential back-off.
            print(f'  Attempt {attempt + 1} failed: {e}. Retrying in {delay:.1f}s...')
            time.sleep(delay)
        except Exception as e:
            raise
    raise ToolTimeoutError(f'All {max_retries} attempts failed. Last error: {last_error}')

# Simulate a slow Walmart inventory API
def slow_inventory_api(sku: str) -> str:
    time.sleep(3.0)  # Exceeds the 2s timeout
    return f'SKU {sku}: In stock'

def fast_inventory_api(sku: str) -> str:
    time.sleep(0.1)  # Well within timeout
    return f'SKU {sku}: In stock (24 units)'

print('Test 1: Slow API (should timeout and retry)...')
try:
    retry_with_backoff(slow_inventory_api, {'sku': 'GV-MILK-1G'}, max_retries=2, base_delay=0.2)
except ToolTimeoutError as e:
    print(f'  All retries exhausted: {e}')
    print('  Fallback: serving cached inventory data')

print()
print('Test 2: Fast API (should succeed)...')
result = retry_with_backoff(fast_inventory_api, {'sku': 'GV-MILK-1G'})
print(f'  Result: {result}')

Test 1: Slow API (should timeout and retry)...
  Attempt 1 failed: Tool call timed out after 2.0s. Retrying in 0.2s...
  Attempt 2 failed: Tool call timed out after 2.0s. Retrying in 0.4s...
  All retries exhausted: All 2 attempts failed. Last error: Tool call timed out after 2.0s
  Fallback: serving cached inventory data

Test 2: Fast API (should succeed)...
  Result: SKU GV-MILK-1G: In stock (24 units)


In [ ]:
# 0.5s → 1s → 2s → 4s

# 0.6s → 0.9s → 2.3s → 3.7s

### Failure Mode 2: Hallucination Detection

In [9]:
VALID_SKUS = {'GV-MILK-1G', 'GV-BREAD-20', 'GV-EGGS-12', 'GV-BUTT-1', 'GV-CHKN-3'}

def detect_hallucinated_sku(llm_extracted_sku: str) -> dict:
    sku = llm_extracted_sku.strip().upper()
    is_valid = sku in VALID_SKUS
    return {
        'sku': sku,
        'valid': is_valid,
        'action': 'proceed' if is_valid else 'reprompt',
        'message': f'SKU {sku} is valid.' if is_valid else f'SKU {sku} not in catalog. Valid SKUs: {sorted(VALID_SKUS)}',
    }

def sku_grounded_response(query: str) -> str:
    # Step 1: LLM extracts a SKU
    extract_prompt = ('From this customer query, extract the product SKU if mentioned, '
                      'or infer the most likely SKU from: GV-MILK-1G, GV-BREAD-20, GV-EGGS-12, GV-BUTT-1, GV-CHKN-3. '
                      'Return only the SKU string, nothing else.')
    extracted = llm.invoke([SystemMessage(content=extract_prompt), HumanMessage(content=query)])
    sku_candidate = extracted.content.strip()

    # Step 2: Validate -- catch hallucinated SKUs before they hit the inventory API
    check = detect_hallucinated_sku(sku_candidate)
    print(f'  Extracted SKU : {sku_candidate}')
    print(f'  Validation    : {check["message"]}')
    print(f'  Action        : {check["action"]}')
    if check['action'] == 'reprompt':
        return 'I could not find that product in our catalog. Could you clarify which item you mean?'

    # Step 3: Only call the inventory API with a validated SKU
    inventory = {
        'GV-MILK-1G': 'In stock: 24 units', 'GV-CHKN-3': 'Out of stock',
    }
    return inventory.get(check['sku'], 'Inventory data not available.')

print('Test 1: Valid product query')
print('  Result:', sku_grounded_response('Is milk available?'))
print()
print('Test 2: Ambiguous query (LLM may hallucinate SKU)')
print('  Result:', sku_grounded_response('Do you have the XYZ-9999 item?'))

Test 1: Valid product query
  Extracted SKU : GV-MILK-1G
  Validation    : SKU GV-MILK-1G is valid.
  Action        : proceed
  Result: In stock: 24 units

Test 2: Ambiguous query (LLM may hallucinate SKU)
  Extracted SKU : GV-MILK-1G
  Validation    : SKU GV-MILK-1G is valid.
  Action        : proceed
  Result: In stock: 24 units


### Failure Mode 3: Circuit Breaker Pattern

A circuit breaker stops calling a failing service after N consecutive failures.
It moves through three states: CLOSED (normal) → OPEN (blocking) → HALF-OPEN (testing).

This prevents cascading failures when a Walmart microservice is degraded.

In [ ]:
# If a Walmart pricing service is failing repeatedly, the agent should not keep calling it again and again. After a certain number of failures, it should temporarily stop sending requests, wait for recovery time, test the service once, and only then resume normal traffic.

class CircuitBreaker:
    def __init__(self, name: str, failure_threshold: int = 3, recovery_timeout: float = 5.0):
        self.name              = name
        self.failure_threshold = failure_threshold
        self.recovery_timeout  = recovery_timeout
        self.failure_count     = 0
        self.state             = 'CLOSED'   # CLOSED | OPEN | HALF-OPEN
        self.last_failure_time = 0.0

    def call(self, fn, *args, **kwargs):
        if self.state == 'OPEN':
            elapsed = time.time() - self.last_failure_time # Calculate how long the circuit has been open
            if elapsed >= self.recovery_timeout:
                self.state = 'HALF-OPEN'
                print(f'  [{self.name}] Circuit HALF-OPEN -- testing recovery')
            else:
                raise RuntimeError(f'[{self.name}] Circuit OPEN. Retry in {self.recovery_timeout - elapsed:.1f}s')

        try:
            result = fn(*args, **kwargs)
            if self.state == 'HALF-OPEN':
                self.state = 'CLOSED'
                self.failure_count = 0
                print(f'  [{self.name}] Circuit CLOSED -- service recovered')
            return result
        except Exception as e:
            self.failure_count += 1
            self.last_failure_time = time.time()
            if self.failure_count >= self.failure_threshold:
                self.state = 'OPEN'
                print(f'  [{self.name}] Circuit OPEN after {self.failure_count} failures')
            raise

# Simulate a flaky Walmart pricing API
call_count = [0]
def flaky_pricing_api(sku: str) -> str:
    call_count[0] += 1
    # Fails on first 3 calls, recovers on call 4+
    if call_count[0] <= 3:
        raise ConnectionError(f'Pricing service unavailable (call #{call_count[0]})')
    return f'SKU {sku}: $3.98'

cb = CircuitBreaker('WalmartPricingAPI', failure_threshold=3, recovery_timeout=2.0)

print('Simulating 6 calls to flaky pricing API:')
for i in range(6):
    try:
        result = cb.call(flaky_pricing_api, 'GV-MILK-1G')
        print(f'  Call {i+1}: SUCCESS -- {result}')
    except RuntimeError as e:
        print(f'  Call {i+1}: BLOCKED -- {e}')
    except ConnectionError as e:
        print(f'  Call {i+1}: FAILED  -- {e} | Circuit state: {cb.state}')
    if i == 3:
        print('  (Waiting for recovery timeout...)')
        time.sleep(2.1)  # Allow circuit to move to HALF-OPEN

Simulating 6 calls to flaky pricing API:
  Call 1: FAILED  -- Pricing service unavailable (call #1) | Circuit state: CLOSED
  Call 2: FAILED  -- Pricing service unavailable (call #2) | Circuit state: CLOSED
  [WalmartPricingAPI] Circuit OPEN after 3 failures
  Call 3: FAILED  -- Pricing service unavailable (call #3) | Circuit state: OPEN
  Call 4: BLOCKED -- [WalmartPricingAPI] Circuit OPEN. Retry in 2.0s
  (Waiting for recovery timeout...)
  [WalmartPricingAPI] Circuit HALF-OPEN -- testing recovery
  [WalmartPricingAPI] Circuit CLOSED -- service recovered
  Call 5: SUCCESS -- SKU GV-MILK-1G: $3.98
  Call 6: SUCCESS -- SKU GV-MILK-1G: $3.98


## Summary

| Topic | Key Takeaway |
|---|---|
| Timeout | Always set a budget; retry with exponential back-off; cache for fallback |
| Hallucination | Validate all structured LLM outputs before passing them to APIs |
| Circuit breaker | Wrap every external service call; open after 3 failures; test with half-open |

These three patterns (timeout + retry, hallucination guard, circuit breaker) are the minimum
resilience baseline for any Walmart production agent.